# ROGII Wellbore Geology Prediction - Model Development

This notebook focuses on model training, hyperparameter tuning, and evaluation.

## Contents
1. Setup and Data Loading
2. Baseline Model Training
3. Hyperparameter Tuning
4. Model Comparison
5. Feature Importance Analysis
6. Cross-Validation Results
7. Model Ensemble

## 1. Setup and Data Loading

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data import load_all_data
from src.preprocess import preprocess_data, add_features
from src.model import train_model, save_model
from src.utils import set_random_seed, calculate_feature_importance
from config import DATA_DIR, MODELS_DIR, RANDOM_SEED

# Set random seed
set_random_seed(RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")

In [ ]:
# Load and preprocess data
print("Loading data...")
train_df = load_all_data(str(DATA_DIR), 'train')
print(f"Loaded {len(train_df)} rows from {train_df['well_id'].nunique()} wells")

print("\nPreprocessing...")
train_processed, scaler = preprocess_data(train_df, is_train=True)

print("\nFeature engineering...")
train_featured = add_features(train_processed)
print(f"Final shape: {train_featured.shape}")

In [ ]:
# Prepare features and target
target_col = 'TVT'
exclude_cols = ['TVT', 'well_id', 'id']
feature_cols = [col for col in train_featured.columns if col not in exclude_cols]

# Handle missing values
train_featured[feature_cols] = train_featured[feature_cols].fillna(0)

X = train_featured[feature_cols]
y = train_featured[target_col]
groups = train_featured['well_id']

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X)}")
print(f"Wells: {groups.nunique()}")
print(f"Target range: [{y.min():.4f}, {y.max():.4f}]")

## 2. Baseline Model Training

In [ ]:
# Train baseline LightGBM model
print("Training baseline LightGBM model...")
lgb_model, lgb_cv_scores = train_model(X, y, groups, model_type='lgb')

print(f"\nCV RMSE Scores: {lgb_cv_scores}")
print(f"Mean CV RMSE: {lgb_cv_scores.mean():.4f} ± {lgb_cv_scores.std():.4f}")

In [ ]:
# Visualize CV scores
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(lgb_cv_scores) + 1), lgb_cv_scores, alpha=0.7)
plt.axhline(lgb_cv_scores.mean(), color='r', linestyle='--', label=f'Mean: {lgb_cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('RMSE')
plt.title('LightGBM Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Feature Importance Analysis

In [ ]:
# Calculate feature importance
importance_df = calculate_feature_importance(lgb_model, feature_cols, top_n=30)
print(importance_df.head(20))

In [ ]:
# Plot feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(len(importance_df)), importance_df['importance_pct'])
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance (%)')
plt.title('Top 30 Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Model Comparison

In [ ]:
# Train XGBoost
print("Training XGBoost model...")
xgb_model, xgb_cv_scores = train_model(X, y, groups, model_type='xgb')
print(f"XGBoost Mean CV RMSE: {xgb_cv_scores.mean():.4f} ± {xgb_cv_scores.std():.4f}")

In [ ]:
# Train CatBoost
print("Training CatBoost model...")
cat_model, cat_cv_scores = train_model(X, y, groups, model_type='cat')
print(f"CatBoost Mean CV RMSE: {cat_cv_scores.mean():.4f} ± {cat_cv_scores.std():.4f}")

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': ['LightGBM', 'XGBoost', 'CatBoost'],
    'Mean RMSE': [lgb_cv_scores.mean(), xgb_cv_scores.mean(), cat_cv_scores.mean()],
    'Std RMSE': [lgb_cv_scores.std(), xgb_cv_scores.std(), cat_cv_scores.std()]
})

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Plot comparison
plt.figure(figsize=(10, 6))
x_pos = range(len(comparison_df))
plt.bar(x_pos, comparison_df['Mean RMSE'], yerr=comparison_df['Std RMSE'], 
        alpha=0.7, capsize=10)
plt.xticks(x_pos, comparison_df['Model'])
plt.ylabel('RMSE')
plt.title('Model Comparison (Mean CV RMSE with Std Dev)')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Model Ensemble

In [ ]:
# Simple averaging ensemble
lgb_pred = lgb_model.predict(X)
xgb_pred = xgb_model.predict(X)
cat_pred = cat_model.predict(X)

ensemble_pred = (lgb_pred + xgb_pred + cat_pred) / 3

# Calculate ensemble RMSE on training data (for reference only)
from sklearn.metrics import mean_squared_error
ensemble_rmse = np.sqrt(mean_squared_error(y, ensemble_pred))
print(f"Ensemble Training RMSE: {ensemble_rmse:.4f}")
print("Note: This is training RMSE, not CV RMSE. Use with caution.")

## 6. Save Best Model

In [ ]:
# Determine best model
best_model_name = comparison_df.loc[comparison_df['Mean RMSE'].idxmin(), 'Model']
print(f"Best model: {best_model_name}")

# Save best model
if best_model_name == 'LightGBM':
    best_model = lgb_model
    model_type = 'lgb'
elif best_model_name == 'XGBoost':
    best_model = xgb_model
    model_type = 'xgb'
else:
    best_model = cat_model
    model_type = 'cat'

model_path = MODELS_DIR / f"best_model_{model_type}.pkl"
save_model(best_model, str(model_path))
print(f"Best model saved to {model_path}")

## Summary

- Trained and compared LightGBM, XGBoost, and CatBoost models
- Analyzed feature importance
- Created simple ensemble
- Saved best performing model

Next steps:
1. Hyperparameter tuning for best model
2. Advanced feature engineering
3. Generate predictions on test set